In [42]:
import pennylane as qml
import jax
import jax.numpy as jnp

In [43]:
num_qubits = 2
weightage = [1.0,-0.5,1.5]
observable = [qml.PauliX(0),qml.PauliZ(1),qml.PauliY(0)@qml.PauliY(1)]
hamil = qml.Hamiltonian(weightage,observable)
hamil_matrix = qml.matrix(hamil)

In [44]:
device = qml.device('default.qubit',wires = num_qubits)
@qml.qnode(device,interface='jax')
def state(theta):
    qml.RX(theta[0],wires = 0)
    qml.RZ(theta[1],wires = 1)
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0,1])
    qml.RX(theta[2],wires = 1)
    qml.RY(theta[3],wires = 0)
    return qml.state()

In [45]:
def compute_M_and_V(theta):
    states = state(theta)
    derivative = jax.jacfwd(state)(theta)
    M = jnp.real(jnp.dot(derivative.conj().T,derivative))
    new_states = jnp.dot(hamil_matrix,states)
    V = jnp.dot(new_states.conj().T,derivative)
    V = jnp.imag(V)
    return (M,V)

In [46]:
dt = 0.02
theta = jnp.array([0.1,0.3,1.0,0.95])

In [47]:
for i in range(50):
    (m,v) = compute_M_and_V(theta)
    m_reg = m+1e-6*jnp.eye(4)
    theta_dot = jnp.linalg.solve(m_reg,v)
    theta = theta+dt*theta_dot
print(f'final paramter is {theta}')

final paramter is [-0.1977496  2.8348866 -0.6286893  0.4731471]
